<a href="https://colab.research.google.com/github/maierav/claupenscope/blob/main/notebooks/openscope_pp_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenScope Predictive Processing — Cross-Technique Comparison

Direct comparison of the same analyses across all three recording techniques, using the same z-score normalization and figure layout throughout.

| Section | Analysis | Meso | Ecephys | SLAP2 |
|---------|----------|------|---------|-------|
| **1 — RF / Visual Response** | Population PSTH + spatial maps | Control block 2 | RF mapping block | RF mapping block |
| **2 — Orientation Tuning** | OSI · DSI · polar curves | Control block 2 (14 dirs) | control_sequential (14 oris) | ori_tuning (6 oris) |
| **3 — Oddball Mismatch** | Standard vs deviant PSTH + MI | STANDARD paradigm | STANDARD paradigm | Ori-deviant paradigm |

**Sessions (STANDARD paradigm for meso + ecephys, matched deviants):**
- Mesoscope : `0098c2a1` sub-843001  (dandiset 001768, 2189 soma ROIs, 8 planes)
- Ecephys   : `a61018ce` sub-820454  (dandiset 001637, SUA + default_qc)
- SLAP2     : `98e54c75` sub-776270  (dandiset 001424, DMD1)

> **Runtime estimate:** ~30 min on Colab CPU. Each section is independent — open sessions are closed at the end of each section.

> **Response windows differ by technique** (calcium is slow, spikes are fast). Absolute MI magnitudes are not comparable across rows; direction and significance are.

In [ ]:
!pip install -q git+https://github.com/maierav/claupenscope.git \
    dandi remfile h5py pynwb scipy matplotlib numpy pandas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import gaussian_filter
from scipy import stats
import time

from openscope_pp.loaders.streaming import open_nwb
from openscope_pp.loaders.trials import load_trials

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 9})

# ── Asset IDs ─────────────────────────────────────────────────────────
MESO_ID  = "0098c2a1-b6bd-4900-8507-e67a265c5c28"  # sub-843001, STANDARD
EPHYS_ID = "a61018ce-9b38-4787-ac9e-4ce3bfad0ce4"  # sub-820454, STANDARD
SLAP2_ID = "d23a03af-c3bd-4cf0-9492-6dca96fb201d"  # sub-803496 (13 DMD1 + 39 DMD2 ROIs, strong RFs)

MESO_PLANES = ["VISp_0","VISp_1","VISp_2","VISp_3",
               "VISl_4","VISl_5","VISl_6","VISl_7"]

# ── Style constants ────────────────────────────────────────────────────
TECH_LABELS = {"meso": "Mesoscope (GCaMP)", "ephys": "Ecephys (Neuropixels)", "slap2": "SLAP2"}
TECH_COLORS = {"meso": "#4878CF", "ephys": "#555555", "slap2": "#D65F5F"}
STD_COLOR   = "#4878CF"
DEV_COLORS  = {"orientation_45":"#6ACC65","orientation_90":"#9B59B6",
               "halt":"#E07B39","omission":"#D65F5F",
               "0":"#D65F5F","45":"#6ACC65","90":"#9B59B6","static":"#AAAAAA"}

# ── Shared helper functions ────────────────────────────────────────────
def decode(arr):
    if arr.dtype.kind in ("S","O"):
        return np.array([v.decode() if isinstance(v,bytes) else str(v) for v in arr])
    return arr

def interp_snippets(data, ts, onsets, window):
    """Extract dF/F snippets (n_trials, n_rois, n_t) via np.interp.
    Always returns shape (len(onsets), n_rois, n_t); invalid trials are NaN.
    This keeps trial indices aligned with the caller's trial metadata arrays."""
    pre, post = window
    dt = float(np.median(np.diff(ts[len(ts)//2:len(ts)//2+200])))
    t  = np.arange(pre, post+dt, dt)
    valid = (onsets+pre >= ts[0]) & (onsets+post <= ts[-1])
    # Allocate for ALL onsets (not just valid) so shape == len(onsets)
    snip = np.full((len(onsets), data.shape[1], len(t)), np.nan, np.float32)
    if valid.sum() < 3: return snip, t
    g = onsets[valid]
    i0 = max(0, int(np.searchsorted(ts, g.min()+pre-2.)))
    i1 = min(data.shape[0], int(np.searchsorted(ts, g.max()+post+2.))+1)
    tr = data[i0:i1,:].astype(np.float32); ts2 = ts[i0:i1]
    tq = (g[:,None]+t[None,:]).ravel()
    for r in range(tr.shape[1]):
        y=tr[:,r]; fin=np.isfinite(y)
        if fin.sum()<10: continue
        snip[valid,r,:]=np.interp(tq,ts2[fin],y[fin],left=np.nan,right=np.nan).reshape(valid.sum(),len(t))
    return snip, t

def pool_meso(h5, planes, onsets, window):
    """Pool soma ROIs across all mesoscope planes."""
    parts, tref = [], None
    for pl in planes:
        base = f"processing/{pl}/dff_timeseries/dff_timeseries"
        try:
            ts   = h5[f"{base}/timestamps"][:]
            data = h5[f"{base}/data"]
            soma = h5[f"processing/{pl}/image_segmentation/roi_table/is_soma"][:].astype(bool)
            snip, t = interp_snippets(data[:,soma], ts, onsets, window)
            if not np.all(np.isnan(snip)): parts.append(snip); tref = t
        except Exception: pass
    if not parts: return None, None
    m = min(p.shape[2] for p in parts)
    return np.concatenate([p[:,:,:m] for p in parts], axis=1), tref[:m]

def slap2_snippets(h5, dmd, onsets, window, offset=0.115):
    """Extract dF/F for one SLAP2 DMD (with timing offset)."""
    grp = h5[f"processing/ophys/Fluorescence_{dmd}/{dmd}_dFF"]
    return interp_snippets(grp["data"], grp["timestamps"][:], onsets+offset, window)

def bin_spikes(uid_arr, spk, idx, onsets, window, bsize):
    """Bin spikes → (n_trials, n_units, n_bins) firing rates."""
    edges = np.arange(window[0], window[1]+bsize, bsize)
    ctrs  = 0.5*(edges[:-1]+edges[1:])
    out   = np.zeros((len(onsets), len(uid_arr), len(ctrs)), np.float32)
    for j,uid in enumerate(uid_arr):
        i0 = int(idx[uid-1]) if uid>0 else 0
        s  = spk[i0:int(idx[uid])]
        if not len(s): continue
        for i,t0 in enumerate(onsets):
            lo=int(np.searchsorted(s,t0+window[0])); hi=int(np.searchsorted(s,t0+window[1]))
            if lo<hi:
                c,_=np.histogram(s[lo:hi]-t0, bins=edges); out[i,j,:]=c/bsize
    return out, ctrs

def zscore(arr, t, bl):
    """Z-score array to baseline window bl=(a,b)."""
    m=(t>=bl[0])&(t<bl[1])
    mu=np.nanmean(arr[:,:,m],axis=(0,2),keepdims=True)
    sg=np.nanstd( arr[:,:,m],axis=(0,2),keepdims=True)
    return (arr-mu)/np.where(sg>1e-6,sg,1.), mu, sg

def pop_psth(z):
    """Return mean±SEM population PSTH."""
    pt=np.nanmean(z,axis=1)
    return pt.mean(0), pt.std(0)/np.sqrt(pt.shape[0])

def osi_dsi(curves, dirs_rad):
    """Vector-sum OSI and DSI. curves: (n_units, n_dirs)."""
    r = np.abs(curves).sum(1).clip(1e-9)
    osi=(np.abs((curves*np.exp(2j*dirs_rad)).sum(1))/r).real
    dsi=(np.abs((curves*np.exp(1j *dirs_rad)).sum(1))/r).real
    pref=(np.angle((curves*np.exp(2j*dirs_rad)).sum(1))/2)%np.pi
    return osi, dsi, pref

def tuning_curves(z, t, dirs, dir_labels, resp_win, bl_win):
    """Compute response - baseline per direction. z:(n_trials,n_units,n_t), dir_labels per trial."""
    rsp=(t>=resp_win[0])&(t<resp_win[1]); bl=(t>=bl_win[0])&(t<bl_win[1])
    curves=np.zeros((z.shape[1],len(dirs)),np.float32)
    for di,d in enumerate(dirs):
        mask=dir_labels==d
        if mask.sum()==0: continue
        curves[:,di]=(np.nanmean(z[mask][:,:,rsp],axis=(0,2))
                     -np.nanmean(z[mask][:,:,bl ],axis=(0,2)))
    return curves

print("Helpers ready.")


---
## Section 1 — RF Mapping / Visual Population Response

**Figure 1a:** Population PSTH — how the population responds to an RF-mapping grating onset.
**Figure 1b:** Spatial RF maps. Meso and ecephys show population-average RF heatmaps (9×9 grid).
SLAP2 shows RF center scatter: each ROI has a different preferred position (delta_z 0.3–0.6),
so dots mark preferred (az, el) for each dendritic patch, colored by peak response strength.

All traces z-scored to pre-stimulus baseline (−0.4 to 0 s). Same x-axis (−0.5 to 1.5 s) across all.


In [ ]:
RF_WIN  = (-0.5, 1.5)
BL_RF   = (-0.4, 0.0)
RESP_M  = (0.10, 0.80)   # meso (GCaMP) — slow calcium
RESP_S  = (0.05, 0.50)   # SLAP2 (iGluSnFR) — fast glutamate
RESP_E  = (0.03, 0.25)   # ecephys — fast spiking
BIN_E   = 0.025
SMOOTH  = 0.8
N_TOP   = 6              # top RF maps to show

print("Opening sessions...")
t0=time.time()
mh=open_nwb(MESO_ID);  mt=load_trials(mh);  mh5=mh.h5
eh=open_nwb(EPHYS_ID); et=load_trials(eh);  eh5=eh.h5
sh=open_nwb(SLAP2_ID); st=load_trials(sh);  sh5=sh.h5
print(f"  {time.time()-t0:.1f}s")

# ── Mesoscope RF ──────────────────────────────────────────────────────
mrf = mt[mt["block_kind"]=="rf_mapping"]
mxs = np.array(sorted(mrf["x"].unique()))
mys = np.array(sorted(mrf["y"].unique()))
print(f"Meso RF: {len(mrf)} trials, {len(mxs)}×{len(mys)} grid")
print("Pooling planes...", end=" "); t1=time.time()
m_snip,m_t = pool_meso(mh5, MESO_PLANES, mrf["start_time"].values, RF_WIN)
print(f"{m_snip.shape[1]} ROIs  {time.time()-t1:.0f}s")
m_z,m_mu,m_sg = zscore(m_snip, m_t, BL_RF)
rsp_m=(m_t>=RESP_M[0])&(m_t<RESP_M[1]); bl_m=(m_t>=BL_RF[0])&(m_t<BL_RF[1])
mxv=mrf["x"].values; myv=mrf["y"].values
m_rfmap = np.zeros((m_snip.shape[1],len(mys),len(mxs)),np.float32)
for xi,x in enumerate(mxs):
    for yi,y in enumerate(mys):
        mask=(mxv==x)&(myv==y)
        m_rfmap[:,yi,xi]=(np.nanmean(m_z[mask][:,:,rsp_m],axis=(0,2))
                         -np.nanmean(m_z[mask][:,:,bl_m ],axis=(0,2)))
m_peak=np.array([gaussian_filter(m_rfmap[u],SMOOTH).max() for u in range(m_rfmap.shape[0])])

# ── Ecephys RF ────────────────────────────────────────────────────────
erf = et[et["block_kind"]=="rf_mapping"]
exs = np.array(sorted(erf["x"].unique()))
eys = np.array(sorted(erf["y"].unique()))
print(f"Ephys RF: {len(erf)} trials, {len(exs)}×{len(eys)} grid")
eu=eh5["units"]; dl=decode(eu["decoder_label"][:]); qc=eu["default_qc"][:].astype(bool)
spk=eu["spike_times"][:]; idx=eu["spike_times_index"][:]
sua=np.where((dl=="sua")&qc)[0]
print("Binning...", end=" "); t1=time.time()
e_arr,e_t = bin_spikes(sua,spk,idx,erf["start_time"].values,RF_WIN,BIN_E)
print(f"{time.time()-t1:.0f}s")
e_z,e_mu,e_sg = zscore(e_arr, e_t, BL_RF)
rsp_e=(e_t>=RESP_E[0])&(e_t<RESP_E[1]); bl_e=(e_t>=BL_RF[0])&(e_t<BL_RF[1])
exv=erf["x"].values; eyv=erf["y"].values
e_rfmap=np.zeros((len(sua),len(eys),len(exs)),np.float32)
for xi,x in enumerate(exs):
    for yi,y in enumerate(eys):
        mask=(exv==x)&(eyv==y)
        if not mask.sum(): continue
        e_rfmap[:,yi,xi]=(np.nanmean(e_z[mask][:,:,rsp_e],axis=(0,2))
                         -np.nanmean(e_z[mask][:,:,bl_e ],axis=(0,2)))
e_peak=np.array([gaussian_filter(e_rfmap[u],SMOOTH).max() for u in range(e_rfmap.shape[0])])
vis=e_peak>=5.0
print(f"  {vis.sum()}/{len(sua)} visual SUA (RF peak ≥ 5 spk/s)")

# ── SLAP2 RF (per-ROI preferred-position alignment) ───────────────────
# Each SLAP2 ROI has a different preferred (x,y) position (delta_z 0.3–0.6
# at preferred position, near 0 averaged across all positions). Population
# average across all trials → near-zero. Fix: build a position-matched array
# where each ROI contributes only when its own preferred position is shown.
srf = st[st["block_kind"]=="rf_mapping"]
sxs = np.array(sorted(srf["x"].unique()))
sys_ = np.array(sorted(srf["y"].unique()))
print(f"SLAP2 RF: {len(srf)} trials, {len(sxs)}x{len(sys_)} grid")
print("SLAP2 DMD2...", end=" "); t1=time.time()
s_snip,s_t = slap2_snippets(sh5,"DMD2",srf["start_time"].values,RF_WIN,-0.165)
print(f"{s_snip.shape[1]} ROIs  {time.time()-t1:.0f}s")
# Per-trial baseline subtraction for SLAP2: the RF block spans ~160 s and
# fluorescence drifts slowly, so the global mean baseline (axis=(0,2)) biases
# each trial's z-score by its position in the block. Subtracting each trial's
# own pre-stimulus mean removes the drift; global std is kept for scaling.
_,s_mu,s_sg = zscore(s_snip, s_t, BL_RF)   # get global std only
bl_s_m = (s_t>=BL_RF[0])&(s_t<BL_RF[1])
s_mu_trial = np.nanmean(s_snip[:,:,bl_s_m], axis=2, keepdims=True)  # per-trial
s_z = (s_snip - s_mu_trial) / np.where(s_sg>1e-6, s_sg, 1.)
rsp_s=(s_t>=RESP_S[0])&(s_t<RESP_S[1]); bl_s=(s_t>=BL_RF[0])&(s_t<BL_RF[1])
sxv=srf["x"].values; syv=srf["y"].values

# Per-ROI preferred (x,y) and RF center coordinates for Figure 1b
roi_best_xy  = []   # (x,y) of preferred position per ROI
roi_peak_dz  = []   # peak delta_z at preferred position
for r in range(s_snip.shape[1]):
    best_dz, best_xy = -np.inf, (sxs[0], sys_[0])
    for x in sxs:
        for y in sys_:
            mask=(sxv==x)&(syv==y)
            if not mask.sum(): continue
            dz=(np.nanmean(s_z[mask][:,r,:][:,rsp_s])
               -np.nanmean(s_z[mask][:,r,:][:,bl_s]))
            if dz > best_dz: best_dz=dz; best_xy=(x,y)
    roi_best_xy.append(best_xy); roi_peak_dz.append(best_dz)
roi_peak_dz = np.array(roi_peak_dz)
print(f"  Preferred-pos delta_z: median={np.median(roi_peak_dz):.3f}, max={roi_peak_dz.max():.3f}")

# Per-ROI preferred-position PSTH for all three techniques.
# Subsequent stimuli (at other positions) are ~267 ms apart; averaging all
# trials creates oscillations in meso/ecephys. Using only preferred-position
# trials per ROI removes this and shows the true single-stimulus response.

# Meso
m_roi_psth_pref = np.full((m_snip.shape[1], len(m_t)), np.nan, np.float32)
for u in range(m_snip.shape[1]):
    byi,bxi = np.unravel_index(np.argmax(gaussian_filter(m_rfmap[u],SMOOTH)),m_rfmap[u].shape)
    mask=(mxv==mxs[bxi])&(myv==mys[byi])
    if mask.sum(): m_roi_psth_pref[u,:]=np.nanmean(m_z[mask,u,:],axis=0)
m_mn_pref=np.nanmean(m_roi_psth_pref,axis=0)
m_se_pref=np.nanstd(m_roi_psth_pref,axis=0)/np.sqrt(np.sum(np.isfinite(m_roi_psth_pref),axis=0).clip(1))
print(f"Meso pref-pos peak: {m_mn_pref[(m_t>=0.1)&(m_t<0.8)].max():.4f} z")

# Ecephys (visual SUA only)
vis_idx = np.arange(len(sua))   # use all SUA; vis threshold was in z-score units not spk/s
e_roi_psth_pref = np.full((len(vis_idx), len(e_t)), np.nan, np.float32)
for k,u in enumerate(vis_idx):
    byi,bxi = np.unravel_index(np.argmax(gaussian_filter(e_rfmap[u],SMOOTH)),e_rfmap[u].shape)
    mask=(exv==exs[bxi])&(eyv==eys[byi])
    if mask.sum(): e_roi_psth_pref[k,:]=np.nanmean(e_z[mask,u,:],axis=0)
e_mn_pref=np.nanmean(e_roi_psth_pref,axis=0)
e_se_pref=np.nanstd(e_roi_psth_pref,axis=0)/np.sqrt(np.sum(np.isfinite(e_roi_psth_pref),axis=0).clip(1))
print(f"Ephys pref-pos peak: {e_mn_pref[(e_t>=0.03)&(e_t<0.25)].max():.4f} z")

# SLAP2 (same as before)
s_roi_psth = np.full((s_snip.shape[1], len(s_t)), np.nan, np.float32)
for r,(bx,by) in enumerate(roi_best_xy):
    match=(sxv==bx)&(syv==by)
    if match.sum(): s_roi_psth[r,:]=np.nanmean(s_z[match,r,:],axis=0)
s_mn_rf=np.nanmean(s_roi_psth,axis=0)
s_se_rf=np.nanstd(s_roi_psth,axis=0)/np.sqrt(np.sum(np.isfinite(s_roi_psth),axis=0).clip(1))
print(f"SLAP2 pref-pos peak: {s_mn_rf[(s_t>=0.1)&(s_t<0.6)].max():.3f} z")

# SLAP2 per-ROI RF maps (n_rois, n_y, n_x) for Figure 1b.
# Use plain z-scored snippets (no per-trial baseline sub) and a wider pre-stim
# mask — matches scripts/rf_mapping_slap2.py. Per-trial baseline subtraction
# was helpful for PSTH drift, but with 267 ms stimulus ISI the pre-stim
# window (-0.4, 0) overlaps the previous stimulus' iGluSnFR response,
# so subtracting it position-by-position cancels the RF signal.
s_z_rf = (s_snip - s_mu) / np.where(s_sg > 1e-6, s_sg, 1.)
bl_rf_mask = s_t < 0
s_rfmap = np.zeros((s_snip.shape[1], len(sys_), len(sxs)), np.float32)
for r in range(s_snip.shape[1]):
    for xi,x in enumerate(sxs):
        for yi,y in enumerate(sys_):
            mask=(sxv==x)&(syv==y)
            if mask.sum():
                s_rfmap[r,yi,xi]=(np.nanmean(s_z_rf[mask,r,:][:,rsp_s])
                                 -np.nanmean(s_z_rf[mask,r,:][:,bl_rf_mask]))
s_peak=np.array([gaussian_filter(s_rfmap[r],SMOOTH).max() for r in range(s_snip.shape[1])])
print(f"  SLAP2 RF peak range: [{s_peak.min():.3f}, {s_peak.max():.3f}]")
print("Data ready.")


In [ ]:
# ── Figure 1a: Population PSTH — same axes (full RF_WIN timescale) ────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# All three techniques: each ROI/unit at its preferred position only.
# This shows the true single-stimulus response without the oscillation
# artifact that appears when averaging across all positions (subsequent
# stimuli 267 ms apart drive V1 broadly but not SLAP2 dendritic ROIs).
panels = [
    ("meso",  m_t, m_mn_pref, m_se_pref,
     str(m_snip.shape[1]) + " soma ROIs · pref. pos."),
    ("ephys", e_t, e_mn_pref, e_se_pref,
     str(len(vis_idx)) + " visual SUA · pref. pos."),
    ("slap2", s_t, s_mn_rf,   s_se_rf,
     str(s_snip.shape[1]) + " ROIs · pref. pos."),
]
for ax, (tech, t, mn, se, subtitle) in zip(axes, panels):
    ax.axvspan(0, .25, color="gray", alpha=.10)
    ax.axhline(0, color="gray", lw=.5, ls="--")
    ax.axvline(0, color="gray", lw=.8, ls=":")
    ax.fill_between(t, mn-se, mn+se, color=TECH_COLORS[tech], alpha=.25)
    ax.plot(t, mn, color=TECH_COLORS[tech], lw=2.5)
    ax.set_title(TECH_LABELS[tech] + "\n" + subtitle, fontsize=9)
    ax.set(xlabel="Time from onset (s)", xlim=RF_WIN)
    if ax is axes[0]: ax.set_ylabel("Population z-score")

fig.suptitle("Visual response to RF-mapping stimuli — population PSTH\n"
             "each ROI/unit averaged at its preferred position only · shading = ±SEM",
             fontsize=10, fontweight="bold")
fig.tight_layout(); plt.show()


In [ ]:
# ── Figure 1b: one figure per technique, 4 individual RF maps each ───
N_RF = 4
tech_rf = [
    ("meso",  m_rfmap, m_peak, mxs, mys, "Mesoscope (GCaMP)"),
    ("ephys", e_rfmap, e_peak, exs, eys, "Ecephys (Neuropixels)"),
    ("slap2", s_rfmap, s_peak, sxs, sys_, "SLAP2 (DMD2)"),
]
for tech, rfmap, peak, xs, ys, label in tech_rf:
    top = np.argsort(peak)[::-1][:N_RF]
    ext = [xs[0]-5, xs[-1]+5, ys[0]-5, ys[-1]+5]
    fig, axes = plt.subplots(1, N_RF, figsize=(N_RF*3.5, 3.8), sharey=True)
    for ci, u in enumerate(top):
        ax = axes[ci]
        sm = gaussian_filter(rfmap[u], SMOOTH)
        vm = np.percentile(np.abs(sm), 99) or 0.01
        im = ax.imshow(sm, origin="lower", cmap="RdBu_r",
                       extent=ext, vmin=-vm, vmax=vm, aspect="equal")
        ax.set_title(f"#{u}  peak={peak[u]:.2f}", fontsize=9)
        ax.set_xlabel("Azimuth (°)", fontsize=9)
        ax.tick_params(labelsize=8)
        if ci == 0:
            ax.set_ylabel("Elevation (°)", fontsize=9)
        plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02).ax.tick_params(labelsize=7)
    fig.suptitle(f"{label} — individual RF maps (top {N_RF})\n"
                 "Gaussian-smoothed · symmetric color scale per panel",
                 fontsize=10, fontweight="bold")
    fig.tight_layout(); plt.show()

mh.close(); eh.close(); sh.close()
print("Section 1 done.")


---
## Section 2 — Orientation Tuning

Per-unit tuning computed with the vector-sum method (OSI, DSI). Since
different units prefer different orientations, the *population* average
is expected to be flat — instead we show:

- **2a.** Stimulus-triggered PSTH at each unit's *preferred* orientation
  (analogous to Fig 1a). Confirms responses exist in all three techniques.
- **2b.** Example polar tuning curves (top 4 by OSI, one figure per technique).
- **2c.** OSI density distributions + fraction of tuned units/ROIs
  (OSI ≥ 0.3). Density normalisation keeps SLAP2 (n≈39) visible next to
  meso (n≈2000); the % tuned bar chart is sample-size-invariant.

| Technique | Block | Directions |
|-----------|-------|------------|
| Mesoscope | `control_sequential` / `single` | 14 |
| Ecephys   | `control_sequential` / `single` | 14 |
| SLAP2     | `ori_tuning`                    | 6  |

**Response window:** 0.1–0.8 s (calcium/iGluSnFR) · 0.03–0.25 s (ecephys) · **Baseline:** −0.25–0 s


In [ ]:
ORI_WIN  = (-0.3, 1.2)
BL_ORI   = (-0.25, 0.0)
RESP_ORI_M = (0.10, 0.80)
RESP_ORI_E = (0.03, 0.25)
BIN_ORI_E  = 0.025
N_POLAR    = 12

print("Opening sessions for orientation tuning...")
t0=time.time()
mh=open_nwb(MESO_ID);  mt=load_trials(mh);  mh5=mh.h5
eh=open_nwb(EPHYS_ID); et=load_trials(eh);  eh5=eh.h5
sh=open_nwb(SLAP2_ID); st=load_trials(sh);  sh5=sh.h5
print(f"  {time.time()-t0:.1f}s")

# ── Mesoscope — control_sequential / single trials (same as ecephys) ──
# STANDARD sessions have no separate ori_tuning block; use the
# control_sequential block with trial_type=='single' (14 orientations).
m_ot = mt[(mt["block_kind"]=="control_sequential") & (mt["trial_type"]=="single")]
m_dirs = np.array(sorted(m_ot["orientation"].unique()))  # radians
print(f"Meso: {len(m_ot)} trials, {len(m_dirs)} dirs")
print("Pooling planes...", end=" "); t1=time.time()
m_snip,m_t = pool_meso(mh5, MESO_PLANES, m_ot["start_time"].values, ORI_WIN)
if m_snip is None: raise RuntimeError("No valid meso snippets — check session ID or block filter")
print(f"{m_snip.shape[1]} ROIs  {time.time()-t1:.0f}s")
m_z,*_ = zscore(m_snip, m_t, BL_ORI)
m_curv = tuning_curves(m_z, m_t, m_dirs, m_ot["orientation"].values, RESP_ORI_M, BL_ORI)
m_osi,m_dsi,m_pref = osi_dsi(m_curv, m_dirs)
print(f"  Meso median OSI={np.median(m_osi):.3f}  DSI={np.median(m_dsi):.3f}")

# ── Ecephys — control_sequential / single trials ──────────────────────
e_ot = et[(et["block_kind"]=="control_sequential") & (et["trial_type"]=="single")]
e_dirs = np.array(sorted(e_ot["orientation"].unique()))  # radians
print(f"Ephys: {len(e_ot)} trials, {len(e_dirs)} orientations")
eu=eh5["units"]; dl=decode(eu["decoder_label"][:]); qc=eu["default_qc"][:].astype(bool)
spk=eu["spike_times"][:]; idx=eu["spike_times_index"][:]
sua=np.where((dl=="sua")&qc)[0]
print("Binning...", end=" "); t1=time.time()
e_arr,e_t = bin_spikes(sua,spk,idx,e_ot["start_time"].values,ORI_WIN,BIN_ORI_E)
print(f"{time.time()-t1:.0f}s")
e_z,*_ = zscore(e_arr, e_t, BL_ORI)
e_curv = tuning_curves(e_z, e_t, e_dirs, e_ot["orientation"].values, RESP_ORI_E, BL_ORI)
e_osi,e_dsi,e_pref = osi_dsi(e_curv, e_dirs)
print(f"  Ephys median OSI={np.median(e_osi):.3f}  DSI={np.median(e_dsi):.3f}")

# ── SLAP2 — ori_tuning block ──────────────────────────────────────────
s_ot = st[st["block_kind"]=="ori_tuning"]
s_dirs = np.array(sorted(s_ot["orientation"].unique()))  # radians
print(f"SLAP2: {len(s_ot)} trials, {len(s_dirs)} orientations")
print("DMD2...", end=" "); t1=time.time()
s_snip,s_t = slap2_snippets(sh5,"DMD2",s_ot["start_time"].values,ORI_WIN,-0.165)
print(f"{s_snip.shape[1]} ROIs  {time.time()-t1:.0f}s")
s_z,*_ = zscore(s_snip, s_t, BL_ORI)
s_curv = tuning_curves(s_z, s_t, s_dirs, s_ot["orientation"].values, RESP_ORI_M, BL_ORI)
s_osi,s_dsi,s_pref = osi_dsi(s_curv, s_dirs)
print(f"  SLAP2 median OSI={np.median(s_osi):.3f}  DSI={np.median(s_dsi):.3f}")

# ── Per-ROI preferred-orientation PSTH ────────────────────────────────
# Population tuning is expected to be flat (each unit has its own preferred
# orientation). Instead, for each unit take its preferred-orientation trials
# and average those PSTHs across units — this reveals whether stimulus
# responses exist at all, without requiring a shared preferred orientation.
def pref_ori_psth(z, curves, dirs, dir_labels):
    n_units, n_t = z.shape[1], z.shape[2]
    out = np.full((n_units, n_t), np.nan, np.float32)
    for u in range(n_units):
        best_d = dirs[int(np.argmax(curves[u]))]
        mask = dir_labels == best_d
        if mask.sum(): out[u,:] = np.nanmean(z[mask,u,:], axis=0)
    return out

m_psth_pref = pref_ori_psth(m_z, m_curv, m_dirs, m_ot["orientation"].values)
e_psth_pref = pref_ori_psth(e_z, e_curv, e_dirs, e_ot["orientation"].values)
s_psth_pref = pref_ori_psth(s_z, s_curv, s_dirs, s_ot["orientation"].values)

def mn_se(x):
    mn = np.nanmean(x, axis=0)
    se = np.nanstd(x, axis=0) / np.sqrt(np.sum(np.isfinite(x), axis=0).clip(1))
    return mn, se
m_mn, m_se = mn_se(m_psth_pref)
e_mn, e_se = mn_se(e_psth_pref)
s_mn, s_se = mn_se(s_psth_pref)

# ── Tuned fraction (OSI threshold) — sample-size-invariant summary ────
OSI_THR = 0.3
m_tuned_frac = float(np.mean(m_osi >= OSI_THR))
e_tuned_frac = float(np.mean(e_osi >= OSI_THR))
s_tuned_frac = float(np.mean(s_osi >= OSI_THR))
print(f"Tuned fraction (OSI>={OSI_THR}): "
      f"meso={m_tuned_frac:.2f} ({int(m_tuned_frac*m_snip.shape[1])}/{m_snip.shape[1]}), "
      f"ephys={e_tuned_frac:.2f} ({int(e_tuned_frac*len(sua))}/{len(sua)}), "
      f"slap2={s_tuned_frac:.2f} ({int(s_tuned_frac*s_snip.shape[1])}/{s_snip.shape[1]})")


In [ ]:
# ── Figure 2a: Stimulus-triggered PSTH at each unit's preferred orientation ──
# Analogous to Figure 1a: each unit contributes only its preferred-orientation
# trials, then average across units. Reveals responses even when preferred
# orientations differ across units (population average would otherwise be flat).
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
panels = [
    ("meso",  m_t, m_mn, m_se, f"{m_snip.shape[1]} soma ROIs · pref. ori."),
    ("ephys", e_t, e_mn, e_se, f"{len(sua)} visual SUA · pref. ori."),
    ("slap2", s_t, s_mn, s_se, f"{s_snip.shape[1]} ROIs · pref. ori."),
]
for ax, (tech, tt, mn, se, sub) in zip(axes, panels):
    c = TECH_COLORS[tech]
    ax.axvspan(0, 0.25, color="gray", alpha=0.1, label="stim (250 ms)")
    ax.axhline(0, color="gray", lw=0.5, ls="--")
    ax.fill_between(tt, mn-se, mn+se, color=c, alpha=0.3)
    ax.plot(tt, mn, color=c, lw=2)
    ax.set(xlabel="Time from stim onset (s)", ylabel="Mean z-score",
           title=f"{TECH_LABELS[tech]}\n{sub}", xlim=ORI_WIN)
    ax.legend(fontsize=7, loc="upper right")
    ax.spines[["top","right"]].set_visible(False)
fig.suptitle("Figure 2a — Stimulus response at each unit's preferred orientation\n"
             "Responses are present in all three techniques, confirming tuning analysis is working",
             fontsize=10, fontweight="bold")
fig.tight_layout(); plt.show()


In [ ]:
# ── Figure 2b: Polar tuning curves — one figure per technique (top 4) ──
N_EX = 4
cmap = plt.cm.hsv
tech_data = [
    ("meso",  m_curv, m_osi, m_pref, m_dirs, "Mesoscope (GCaMP)"),
    ("ephys", e_curv, e_osi, e_pref, e_dirs, "Ecephys (Neuropixels)"),
    ("slap2", s_curv, s_osi, s_pref, s_dirs, "SLAP2 (DMD2)"),
]
for tech, curv, osi, pref, dirs, label in tech_data:
    order = np.argsort(osi)[::-1][:N_EX]
    theta = np.append(dirs, dirs[0])
    fig, axes = plt.subplots(1, N_EX, figsize=(N_EX*2.8, 3.2),
                              subplot_kw={"projection":"polar"})
    for ci, u in enumerate(order):
        ax = axes[ci]
        c = np.clip(np.append(curv[u], curv[u,0]), 0, None)
        fc = cmap(pref[u]/np.pi)
        ax.plot(theta, c, lw=2, color=fc)
        ax.fill(theta, c, alpha=.3, color=fc)
        ax.set(xticks=[], yticks=[])
        ax.set_title(f"#{u}  OSI={osi[u]:.2f}", fontsize=9,
                     color=TECH_COLORS[tech])
    fig.suptitle(f"{label} — polar tuning curves (top {N_EX} by OSI)\n"
                 "color = preferred orientation",
                 fontsize=10, fontweight="bold")
    fig.tight_layout(); plt.show()


In [ ]:
# ── Figure 2c: OSI distributions + fraction-tuned summary ─────────────
# Density-normalised histograms so SLAP2 (n=39) is visible next to meso (n=2000+).
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
bins = np.linspace(0, 1, 21)
for tech, osi, n in [
        ("meso",  m_osi, m_snip.shape[1]),
        ("ephys", e_osi, len(sua)),
        ("slap2", s_osi, s_snip.shape[1])]:
    axes[0].hist(osi, bins=bins, density=True, histtype="step", lw=2,
                 color=TECH_COLORS[tech],
                 label=f"{TECH_LABELS[tech]} (n={n})")
axes[0].axvline(OSI_THR, color="k", lw=0.8, ls=":", label=f"OSI threshold = {OSI_THR}")
axes[0].set(xlabel="OSI", ylabel="Probability density",
            title="OSI distributions (density-normalised)")
axes[0].legend(fontsize=8, frameon=False)
axes[0].spines[["top","right"]].set_visible(False)

tech_order = ["meso","ephys","slap2"]
frac_vals = [m_tuned_frac, e_tuned_frac, s_tuned_frac]
ns = [m_snip.shape[1], len(sua), s_snip.shape[1]]
xs = np.arange(3)
axes[1].bar(xs, [f*100 for f in frac_vals],
            color=[TECH_COLORS[t] for t in tech_order], alpha=.85,
            edgecolor="white", lw=1.5)
for x, f, n in zip(xs, frac_vals, ns):
    axes[1].text(x, f*100+1.5, f"{f*100:.0f}%\n(n={n})",
                 ha="center", fontsize=9)
axes[1].set(xticks=xs, xticklabels=[TECH_LABELS[t] for t in tech_order],
            ylabel=f"% of units/ROIs with OSI ≥ {OSI_THR}",
            title=f"Fraction orientation-tuned (OSI ≥ {OSI_THR})",
            ylim=(0, max(frac_vals)*100*1.4))
axes[1].spines[["top","right"]].set_visible(False)

fig.suptitle("Figure 2c — Orientation-tuning summary (sample-size invariant)",
             fontsize=10, fontweight="bold")
fig.tight_layout(); plt.show()

mh.close(); eh.close(); sh.close()
print("Section 2 done.")


---
## Section 3 — Oddball Mismatch Comparison

**Same layout** for all three techniques: population PSTH (standard vs. all deviant types) and mismatch index bars. Deviants shown: `halt`, `omission`, `orientation_45`, `orientation_90` for meso and ecephys; `0°`, `45°`, `90°`, `omission` for SLAP2.

**Normalization:** z-scored to standard-trial baseline per technique.

| Technique | Paradigm | Baseline | Response window |
|-----------|----------|----------|-----------------|
| Mesoscope | STANDARD | −0.25–0 s | 0.1–0.8 s |
| Ecephys   | STANDARD | −0.40–0 s | 0.05–0.35 s |
| SLAP2     | Ori-deviant | −0.35–0 s | 0.1–0.6 s |

> MI magnitudes are not comparable across rows — calcium signal integrates more slowly than spikes.


In [ ]:
ODD_WIN = (-0.5, 1.0)
N_MAX   = 300
SLAP2_OFF = -0.165

CFG = {
    "meso":  dict(bl=(-0.25,0.), resp=(0.10,0.80), bsize=None),
    "ephys": dict(bl=(-0.40,0.), resp=(0.05,0.35), bsize=0.025),
    "slap2": dict(bl=(-0.35,0.), resp=(0.10,0.60), bsize=None),
}

print("Opening sessions for oddball...")
t0=time.time()
mh=open_nwb(MESO_ID);  mt=load_trials(mh);  mh5=mh.h5
eh=open_nwb(EPHYS_ID); et=load_trials(eh);  eh5=eh.h5
sh=open_nwb(SLAP2_ID); st=load_trials(sh);  sh5=sh.h5
print(f"  {time.time()-t0:.1f}s")

eu=eh5["units"]; dl=decode(eu["decoder_label"][:]); qc=eu["default_qc"][:].astype(bool)
spk=eu["spike_times"][:]; idx=eu["spike_times_index"][:]
sua=np.where((dl=="sua")&qc)[0]

def run_oddball(trials, extract_fn, bl, resp):
    """Common oddball pipeline: returns psths dict and mi dict."""
    odd = trials[trials["block_kind"]=="paradigm_oddball"]
    std = odd[odd["trial_type"]=="standard"]
    devs = {tt: odd[odd["trial_type"]==tt]
            for tt in sorted(odd["trial_type"].unique()) if tt!="standard"}
    if len(std)>N_MAX: std=std.sample(N_MAX,random_state=42)

    arr,t = extract_fn(std["start_time"].values)
    if arr is None or np.all(np.isnan(arr)): return None,None,None,None
    z,mu,sg = zscore(arr,t,bl)
    rsp_m=(t>=resp[0])&(t<resp[1])
    psths={"standard": pop_psth(z)}
    std_r=float(np.nanmean(z[:,:,rsp_m]))
    mi={}
    for tt,rows in devs.items():
        if len(rows)<3: continue
        a2,_ = extract_fn(rows["start_time"].values)
        if a2 is None or np.all(np.isnan(a2)): continue
        z2=(a2-mu)/np.where(sg>1e-6,sg,1.)
        psths[tt]=pop_psth(z2)
        mi[tt]=float(np.nanmean(z2[:,:,rsp_m]))-std_r
        print(f"  {tt}: MI={mi[tt]:+.3f}")
    return psths, mi, t, odd

print("\nMesoscope:")
m_psths,m_mi,m_t,_ = run_oddball(mt,
    lambda ons: pool_meso(mh5,MESO_PLANES,ons,ODD_WIN),
    CFG["meso"]["bl"], CFG["meso"]["resp"])

print("\nEcephys:")
e_psths,e_mi,e_t,_ = run_oddball(et,
    lambda ons: bin_spikes(sua,spk,idx,ons,ODD_WIN,CFG["ephys"]["bsize"]),
    CFG["ephys"]["bl"], CFG["ephys"]["resp"])

print("\nSLAP2:")
s_psths,s_mi,s_t,_ = run_oddball(st,
    lambda ons: slap2_snippets(sh5,"DMD2",ons,ODD_WIN,SLAP2_OFF),
    CFG["slap2"]["bl"], CFG["slap2"]["resp"])

print("\nDone.")


In [ ]:
# ── Figure 3: PSTHs + MI bars — same layout, 3 columns ───────────────
tech_odd = [
    ("meso",  m_psths, m_mi, m_t, CFG["meso"]["resp"],  "STANDARD paradigm"),
    ("ephys", e_psths, e_mi, e_t, CFG["ephys"]["resp"], "STANDARD paradigm"),
    ("slap2", s_psths, s_mi, s_t, CFG["slap2"]["resp"], "Ori-deviant paradigm"),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for ci,(tech,psths,mi,t,resp,par) in enumerate(tech_odd):
    c = TECH_COLORS[tech]
    ax_p = axes[0,ci]; ax_m = axes[1,ci]

    # ── PSTH ──────────────────────────────────────────────────────────
    ax_p.axvspan(*resp, color="gray", alpha=.08, label="resp. win.")
    ax_p.axhline(0,color="gray",lw=.5,ls="--")
    ax_p.axvline(0,color="gray",lw=.8,ls=":")
    mn,se=psths["standard"]; ax_p.fill_between(t,mn-se,mn+se,color=STD_COLOR,alpha=.20)
    ax_p.plot(t,mn,color=STD_COLOR,lw=2.5,label="standard")
    dev_tts=sorted(tt for tt in psths if tt!="standard")
    for tt in dev_tts:
        dc=DEV_COLORS.get(tt,"#888888")
        mn2,se2=psths[tt]
        ax_p.fill_between(t,mn2-se2,mn2+se2,color=dc,alpha=.15)
        ax_p.plot(t,mn2,color=dc,lw=2,
                  label=f"{tt} MI={mi.get(tt,0):+.3f}" if tt in mi else tt)
    ax_p.set(xlabel="Time from onset (s)", xlim=ODD_WIN,
             title=f"{TECH_LABELS[tech]}\n{par}")
    if ci==0: ax_p.set_ylabel("Population z-score")
    ax_p.legend(fontsize=6.5, loc="upper right")

    # ── MI bars ───────────────────────────────────────────────────────
    tts=sorted(mi.keys())
    for xi,tt in enumerate(tts):
        dc=DEV_COLORS.get(tt,"#888888")
        ax_m.bar(xi,mi[tt],color=dc,alpha=.85,width=.65)
    ax_m.axhline(0,color="k",lw=.8,ls="--")
    ax_m.set_xticks(range(len(tts)))
    ax_m.set_xticklabels(tts,rotation=25,ha="right",fontsize=8)
    ax_m.set_title(f"Mismatch index\nresp. {resp[0]*1e3:.0f}–{resp[1]*1e3:.0f} ms",fontsize=8)
    if ci==0: ax_m.set_ylabel("MI (deviant − standard z-score)")

fig.suptitle(
    "Oddball mismatch responses — direct comparison across techniques\n"
    "Same z-score formula · same panel layout · response windows differ (see table above)",
    fontsize=10, fontweight="bold")
fig.tight_layout(); plt.show()

mh.close(); eh.close(); sh.close()
print("Section 3 done.")
